# KLA restoration training — one-shot Colab (clone + Run all)

Trains the residual U-Net for **degraded-image restoration** with added **robustness to real SEM acquisition artifacts**. The training corpus combines:

* the **real DRAM + FinFET SEM structures** committed in the repo at `data/sem_sources/` (curated from the SEMICON India *Drift-Sense* dataset drop), and
* a larger **first-party synthetic** source set, so the corpus has scale as well as realism.

The KLA-faithful forward model (additive Gaussian noise, multiplicative speckle, downsampling) produces the paired NoisyLR. On top of that, training-time **extended augmentation** injects the broader SEM artifact family — beam-spot blur + astigmatism, Poisson shot noise, detector readout noise, vignetting, gamma miscalibration, barrel distortion, charging streaks and raster drift/jitter — so the model learns tolerance to them. Validation stays on the clean KLA-faithful NoisyLR, so the selection metric is still comparable to the strict submission run. Results are pipeline evidence only, **not** official KLA scores.

This notebook is **fully automated**: it clones the repository (no manual upload), builds the combined corpus, trains, freezes `models/best.pth` (the exact checkpoint `run.py` loads for the `.npy` evaluator), evaluates once on held-out sources, exercises the `run.py` `.npy` contract, and downloads the trained artifacts.

**To run:** `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`. No prompts, no edits.


## 1. Record the actual runtime


In [ ]:
import platform, torch
print({
    'python': platform.python_version(),
    'torch': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'cuda': torch.version.cuda,
})
assert torch.cuda.is_available(), 'Enable a GPU runtime: Runtime > Change runtime type > T4 GPU'
!nvidia-smi

## 2. Clone the repository (no manual upload)

Clones the finalized submission branch directly. If the repo is **private**, add a Colab Secret named `GITHUB_TOKEN` (a GitHub PAT with `repo` scope) via the key icon in the left sidebar and enable notebook access. If the repo is public, no token is needed. This is what makes the notebook one-shot.


In [ ]:
import os
OWNER_REPO = 'Veer-WebDev/kla-image-restoration'
BRANCH = 'kla-restoration-submission'
token = ''
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN') or ''
except Exception:
    token = os.environ.get('GITHUB_TOKEN', '')
auth = f'x-access-token:{token}@' if token else ''
clone_url = f'https://{auth}github.com/{OWNER_REPO}.git'
!rm -rf kla-image-restoration
rc = os.system(f'git clone --depth 1 --branch {BRANCH} {clone_url} kla-image-restoration')
assert rc == 0, ('clone failed. If the repo is private, add a Colab Secret named '
                 'GITHUB_TOKEN (PAT with repo scope) and enable notebook access, then re-run.')
os.chdir('kla-image-restoration')
print('cwd', os.getcwd())
!git rev-parse --short HEAD
!ls

## 3. Install dependencies (keep Colab's CUDA torch)

Colab already ships a CUDA-matched PyTorch. We install everything **except** torch so the GPU build is preserved, then confirm CUDA is still available. `scipy` (used by the extended-artifact module) is in `requirements.txt`.


In [ ]:
# Install declared deps but do not touch the pre-installed CUDA torch.
!grep -viE '^\s*(torch|torchvision|torchaudio)\b' requirements.txt > _reqs_no_torch.txt
!cat _reqs_no_torch.txt
!python -m pip install -q -r _reqs_no_torch.txt
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'gpu', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
assert torch.cuda.is_available(), 'CUDA torch was clobbered; restart runtime and re-run.'

## 4. Build the combined corpus (real SEM structures + synthetic sources)

The real curated SEM structures live in `data/sem_sources/` (committed). We add a larger synthetic source set for scale, merge both into one source directory, then materialize KLA-faithful GT/NoisyLR pairs. Source split is by SHA-256 **before** views, so no source leaks across train/val/test.


In [ ]:
import glob, shutil, os
# 1) Real curated SEM structures (already in the repo).
n_sem = len(glob.glob('data/sem_sources/*.png'))
print('real SEM sources:', n_sem)
assert n_sem > 0, 'data/sem_sources is empty; expected committed SEM structures'
# 2) Larger synthetic source set for scale.
!python scripts/generate_clean_sem_sources.py --out data/sources_big --count 160 --size 768 --seed 20260817
# 3) Merge both into one source dir (real + synthetic).
merged = 'data/sources_combined'
!rm -rf $merged
os.makedirs(merged, exist_ok=True)
for src in ('data/sem_sources', 'data/sources_big'):
    for p in glob.glob(f'{src}/*.png'):
        shutil.copy2(p, os.path.join(merged, os.path.basename(p)))
print('combined sources:', len(glob.glob(f'{merged}/*.png')))
# 4) Materialize KLA-faithful pairs into data/kla_sem (matches configs/submission_robust_fast.yaml).
!python scripts/materialize_restoration_data.py --source-dir $merged --out data/kla_sem --seed 20260817 --views-per-source 6 --crop-size 512 --scale 2
!cat data/kla_sem/dataset_card.json 2>/dev/null || true
for split in ('train','val','test'):
    n = len(glob.glob(f'data/kla_sem/{split}/NoisyLR/*.npy'))
    print(split, 'pairs:', n)

## 5. Train the robustness configuration

`configs/submission_robust_fast.yaml` trains on the combined corpus with the extended SEM-artifact augmentation enabled (training only). Model is selected by validation PSNR on the clean KLA-faithful NoisyLR; `data/kla_sem/test` is never inspected or tuned against. On a T4 expect roughly 15–25 min (24 epochs, patch 192). For the full 60-epoch schedule use configs/submission_robust.yaml instead.


In [ ]:
!python train.py --config configs/submission_robust_fast.yaml
!ls -la runs/kla_restoration_robust_fast_seed20260817

## 6. Freeze validation-best weights into the run.py location, evaluate held-out sources once


In [ ]:
RUN='runs/kla_restoration_robust_fast_seed20260817'
!mkdir -p models weights results/submission_robust/examples
# models/best.pth is the exact file run.py loads for the .npy evaluator contract.
!cp $RUN/best.pth models/best.pth
!cp $RUN/best.pth weights/final_model.pth
!cp $RUN/resolved_config.yaml weights/final_model.config.yaml 2>/dev/null || true
!sha256sum models/best.pth | tee models/best.sha256
# LPIPS pulls a small backbone over the network (available during Colab training).
!python evaluate.py --gt-dir data/kla_sem/test/GT --noisy-dir data/kla_sem/test/NoisyLR --checkpoint models/best.pth --output-dir results/submission_robust --save-restored results/submission_robust/examples --split all --eval-mode official || python evaluate.py --gt-dir data/kla_sem/test/GT --noisy-dir data/kla_sem/test/NoisyLR --checkpoint models/best.pth --output-dir results/submission_robust --save-restored results/submission_robust/examples --split all --eval-mode official --no-lpips
!cat results/submission_robust/summary.json

## 7. Robustness check: restore held-out pairs re-degraded with the extended SEM artifacts

This is the point of the robustness training. We take held-out test GT, apply the **extended** SEM-artifact degradations on top of the KLA-faithful NoisyLR, and confirm the model still restores well above the bicubic baseline on this harder, artifact-laden input.


In [ ]:
import glob, numpy as np, torch
import sys; sys.path.insert(0, 'src')
from kla_restore.utils import load_image_float, to_tensor, to_numpy
from kla_restore.degradation import DegradationConfig, degrade, sample_seed
from kla_restore.extended_degradation import ExtendedDegradationConfig, apply_extended
from kla_restore.metrics import compute_metrics
from kla_restore.checkpoint import load_model
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
model, meta = load_model('models/best.pth', map_location=dev)
model = model.to(dev).eval()
deg = DegradationConfig()
ext = ExtendedDegradationConfig.from_dict({
    'enabled': True, 'beam_blur_prob': 1.0, 'shot_noise_prob': 1.0, 'detector_noise_prob': 1.0,
    'vignette_prob': 1.0, 'gamma_prob': 1.0, 'barrel_prob': 1.0, 'charging_prob': 1.0, 'drift_jitter_prob': 1.0,
})
gts = sorted(glob.glob('data/kla_sem/test/GT/*.png'))
m_model, m_bic = [], []
for i, gp in enumerate(gts):
    gt = load_image_float(gp, clip=True, grayscale=True).array
    seed = sample_seed(999, gp, i, 0)
    noisy, _ = degrade(gt, deg, seed)
    noisy, _ = apply_extended(noisy, ext, seed)
    with torch.inference_mode():
        parts = model(to_tensor(noisy)[None].to(dev), target_size=gt.shape[:2], clamp=True, return_parts=True)
    pred = to_numpy(parts['restored'][0].float().cpu())
    base = to_numpy(parts['base'][0].float().clamp(0,1).cpu())
    m_model.append(compute_metrics(pred, gt)['psnr'])
    m_bic.append(compute_metrics(base, gt)['psnr'])
import statistics as st
print(f'artifact-laden PSNR: model {st.mean(m_model):.3f} dB  vs  bicubic {st.mean(m_bic):.3f} dB  (gain +{st.mean(m_model)-st.mean(m_bic):.3f})')
assert st.mean(m_model) > st.mean(m_bic), 'model did not beat bicubic on artifact-laden inputs'
print('robustness check OK')

## 8. Exercise the evaluator-facing run.py `.npy` contract


In [ ]:
!rm -rf submission_smoke
# run.py takes positional <input-dir> <output-dir>, reads .npy recursively, writes matching .npy outputs.
!python run.py data/kla_sem/test/NoisyLR submission_smoke
import numpy as np, glob, os
outs = sorted(glob.glob('submission_smoke/**/*.npy', recursive=True))
ins = sorted(glob.glob('data/kla_sem/test/NoisyLR/**/*.npy', recursive=True))
print('inputs', len(ins), 'outputs', len(outs))
a = np.load(outs[0])
print('sample out', os.path.basename(outs[0]), a.shape, a.dtype, float(a.min()), float(a.max()), 'finite', bool(np.isfinite(a).all()))
assert len(ins) == len(outs), 'filename parity failed'
assert a.ndim in (2, 3) and float(a.min()) >= 0.0 and float(a.max()) <= 1.0 and bool(np.isfinite(a).all()), 'output invariants failed'
print('run.py .npy contract OK')

## 9. Archive artifacts and the trained checkpoint

Downloads a zip containing `models/best.pth` (drop it into the repo at the same path to update the submission), the evaluation summary, and manifests.


In [ ]:
import hashlib, pathlib, datetime
artifact = pathlib.Path('kla_restoration_robust_artifacts.zip')
!rm -f $artifact
!zip -qr $artifact models/best.pth models/best.sha256 weights results/submission_robust submission_smoke data/kla_sem/dataset_card.json data/kla_sem/train_manifest.csv data/kla_sem/val_manifest.csv data/kla_sem/test_manifest.csv
print({'artifact': str(artifact), 'sha256': hashlib.sha256(artifact.read_bytes()).hexdigest(), 'utc_finished': datetime.datetime.now(datetime.UTC).isoformat()})
try:
    from google.colab import files
    files.download(str(artifact))
except Exception as exc:
    print('auto-download unavailable, find the zip in the file browser:', exc)